In [1]:
from pathlib import Path
import json
from collections import Counter
import pandas as pd

DATASET_DIR      = Path("/home/ubuntu/work/somin/evaluation/dataset")
RERUN_JSON       = DATASET_DIR / "goldset_final_6.json"
FINAL_JSON       = DATASET_DIR / "goldset_final.json"
TARGET_SCENARIOS = ["S01", "S02", "S03", "S08", "S15", "S17"]

In [4]:
with RERUN_JSON.open(encoding="utf-8") as f:
    rerun_data = json.load(f)

with FINAL_JSON.open(encoding="utf-8") as f:
    final_data = json.load(f)

print(f"소스({RERUN_JSON.name}): {len(rerun_data)}건")
print(f"goldset_final 기존: {len(final_data)}건")

# TARGET_SCENARIOS 항목 제거
kept = [e for e in final_data if e.get("query_id") not in TARGET_SCENARIOS]
removed = len(final_data) - len(kept)
print(f"제거된 기존 항목({', '.join(TARGET_SCENARIOS)}): {removed}건")

# goldset_final_6.json 은 이미 final_grade / confidence / grade_j1~j3 필드 보유
# → 그대로 사용 (llm_raw_output 만 제외)
sid_order = {f"S{i:02d}": i for i in range(1, 30)}

new_entries = [
    {k: v for k, v in r.items() if k != "llm_raw_output"}
    for r in rerun_data
]

# 병합 → 정렬 → 저장
merged = kept + new_entries
merged.sort(key=lambda x: (sid_order.get(x.get("query_id", ""), 99), x.get("isbn", "")))

with FINAL_JSON.open("w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {len(merged)}건 → {FINAL_JSON}")

소스(goldset_final_6.json): 476건
goldset_final 기존: 1678건
제거된 기존 항목(S01, S02, S03, S08, S15, S17): 476건
저장 완료: 1678건 → /home/ubuntu/work/somin/evaluation/dataset/goldset_final.json


In [5]:
# 통계 표
with FINAL_JSON.open(encoding="utf-8") as f:
    final = json.load(f)

rows = []
all_qids = sorted({e["query_id"] for e in final}, key=lambda x: sid_order.get(x, 99))

for qid in all_qids:
    items  = [e for e in final if e.get("query_id") == qid]
    grades = Counter(e.get("final_grade") for e in items if e.get("label_status") == "success")
    failed = sum(1 for e in items if e.get("label_status") != "success")
    total  = len(items)
    rows.append({
        "query_id": qid,
        "grade_0":  grades.get(0, 0),
        "grade_1":  grades.get(1, 0),
        "grade_2":  grades.get(2, 0),
        "grade_3":  grades.get(3, 0),
        "failed":   failed,
        "total":    total,
        "grade≥2":  grades.get(2, 0) + grades.get(3, 0),
    })

df = pd.DataFrame(rows).set_index("query_id")
print(f"총 항목: {len(final):,}  /  query_id 수: {len(all_qids)}")
print()
print(df.to_string())

insufficient = df[df["grade≥2"] < 3]
print()
if insufficient.empty:
    print("✅ 모든 query_id가 grade≥2 최소 3개 기준 충족")
else:
    print(f"⚠️  기준 미달 query_id: {list(insufficient.index)}")

총 항목: 1,678  /  query_id 수: 21

          grade_0  grade_1  grade_2  grade_3  failed  total  grade≥2
query_id                                                            
S01             6       63        5        1       0     75        6
S02            12       68        3        0       0     83        3
S03            19       49        2        0       0     70        2
S04             7       61        7        1       0     76        8
S05            17       65        1        1       0     84        2
S06             2       53       23        2       0     80       25
S07            11       45       10        1       0     67       11
S08            10       67        8        1       0     86        9
S09            23       56        9        0       0     88        9
S10             5       54       21        5       0     85       26
S11            17       55        6        1       0     79        7
S12             8       34       15       23       0     80       38
S1